In [2]:
# ============================================================
# PACKAGE 8.3 : FEATURE ENCODING + FINAL MODEL MATRIX PREPARATION
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import joblib


print("="*75)
print("PACKAGE 8.3 : FEATURE ENCODING + FINAL MODEL MATRIX PREPARATION")
print("="*75)



# ------------------------------------------------------------
# Load Package 8.2 Processed Data
# ------------------------------------------------------------

train = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package2_Leakage_Free_Missing_Value_Handling\Delhi_AQI_Train_Processed.csv"
)

test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package2_Leakage_Free_Missing_Value_Handling\Delhi_AQI_Test_Processed.csv"
)


print("\nProcessed Train Shape:")
print(train.shape)

print("Processed Test Shape:")
print(test.shape)



# ------------------------------------------------------------
# Date Conversion
# ------------------------------------------------------------

train["Date"] = pd.to_datetime(train["Date"])

test["Date"] = pd.to_datetime(test["Date"])



# ------------------------------------------------------------
# Extract Time Features
# ------------------------------------------------------------

def add_time_features(df):

    df = df.copy()

    df["Year"] = df["Date"].dt.year

    df["Month"] = df["Date"].dt.month

    df["Day"] = df["Date"].dt.day

    df["DayOfYear"] = df["Date"].dt.dayofyear


    # Season encoding
    def get_season(month):

        if month in [12,1,2]:
            return "Winter"

        elif month in [3,4,5]:
            return "Summer"

        elif month in [6,7,8,9]:
            return "Monsoon"

        else:
            return "Post_Monsoon"


    df["Season"] = df["Month"].apply(
        get_season
    )


    return df



train = add_time_features(train)

test = add_time_features(test)



# ------------------------------------------------------------
# Separate Target
# ------------------------------------------------------------

TARGET = "AQI_Target"


X_train = train.drop(
    columns=[TARGET]
)

X_test = test.drop(
    columns=[TARGET]
)


y_train = train[TARGET]

y_test = test[TARGET]



# ------------------------------------------------------------
# Remove Date Column
# ------------------------------------------------------------

X_train = X_train.drop(
    columns=["Date"]
)

X_test = X_test.drop(
    columns=["Date"]
)



# ------------------------------------------------------------
# Identify Categorical Columns
# ------------------------------------------------------------

categorical_cols = X_train.select_dtypes(
    include=["object","category","bool"]
).columns


numeric_cols = X_train.select_dtypes(
    include=["int64","float64"]
).columns



print("\nNumeric Features:")
print(len(numeric_cols))


print("\nCategorical Features:")
print(len(categorical_cols))

print(
    list(categorical_cols)
)



# ------------------------------------------------------------
# One Hot Encoding
# FIT ONLY ON TRAIN
# ------------------------------------------------------------

if len(categorical_cols) > 0:


    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )


    X_train_cat = encoder.fit_transform(
        X_train[categorical_cols]
    )


    X_test_cat = encoder.transform(
        X_test[categorical_cols]
    )


    encoded_columns = (
        encoder
        .get_feature_names_out(categorical_cols)
    )


    X_train_cat = pd.DataFrame(
        X_train_cat,
        columns=encoded_columns
    )


    X_test_cat = pd.DataFrame(
        X_test_cat,
        columns=encoded_columns
    )



    X_train_num = X_train.drop(
        columns=categorical_cols
    ).reset_index(drop=True)


    X_test_num = X_test.drop(
        columns=categorical_cols
    ).reset_index(drop=True)



    X_train_final = pd.concat(
        [
            X_train_num,
            X_train_cat
        ],
        axis=1
    )


    X_test_final = pd.concat(
        [
            X_test_num,
            X_test_cat
        ],
        axis=1
    )


else:

    X_train_final = X_train.copy()

    X_test_final = X_test.copy()



# ------------------------------------------------------------
# Reset Index
# ------------------------------------------------------------

X_train_final = X_train_final.reset_index(
    drop=True
)

X_test_final = X_test_final.reset_index(
    drop=True
)



# ------------------------------------------------------------
# Final Check
# ------------------------------------------------------------

print("\n============================================================")
print("FINAL MODEL MATRIX")
print("============================================================")


print("\nX_train Shape:")
print(X_train_final.shape)


print("\nX_test Shape:")
print(X_test_final.shape)


print("\ny_train Shape:")
print(y_train.shape)


print("\ny_test Shape:")
print(y_test.shape)



print("\nRemaining Missing Values:")

print(
    "Train:",
    X_train_final.isnull().sum().sum()
)

print(
    "Test:",
    X_test_final.isnull().sum().sum()
)



# ------------------------------------------------------------
# Save Final Model Files
# ------------------------------------------------------------

X_train_final.to_csv(
    "X_train_final.csv",
    index=False
)

X_test_final.to_csv(
    "X_test_final.csv",
    index=False
)


y_train.to_csv(
    "y_train.csv",
    index=False
)


y_test.to_csv(
    "y_test.csv",
    index=False
)



if len(categorical_cols)>0:

    joblib.dump(
        encoder,
        "AQI_OneHot_Encoder.pkl"
    )



print("\nSaved Files:")
print("1. X_train_final.csv")
print("2. X_test_final.csv")
print("3. y_train.csv")
print("4. y_test.csv")
print("5. AQI_OneHot_Encoder.pkl")


print("\nPackage 8.3 Completed Successfully.")

PACKAGE 8.3 : FEATURE ENCODING + FINAL MODEL MATRIX PREPARATION

Processed Train Shape:
(1826, 82)
Processed Test Shape:
(182, 82)

Numeric Features:
75

Categorical Features:
2
['City', 'Season']

FINAL MODEL MATRIX

X_train Shape:
(1826, 84)

X_test Shape:
(182, 84)

y_train Shape:
(1826,)

y_test Shape:
(182,)

Remaining Missing Values:
Train: 0
Test: 0

Saved Files:
1. X_train_final.csv
2. X_test_final.csv
3. y_train.csv
4. y_test.csv
5. AQI_OneHot_Encoder.pkl

Package 8.3 Completed Successfully.
